<a href="https://colab.research.google.com/github/Gcortes7/Artefactos/blob/main/Menu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


def ejecutar_equipo4():
    ruta = input("Ruta de la carpeta con los CSV: ").strip().strip('"')
    carpeta = Path(ruta).expanduser()

    if not carpeta.is_dir():
        print("No se encontró la carpeta indicada.")
        return

    DATOS = {}
    COLS_BP = ["P7/gamma"]

    for archivo in sorted(carpeta.glob("*.csv")):
        try:
            bp = pd.read_csv(archivo, encoding="utf-8-sig")
            bp.columns = bp.columns.str.strip()
            if "P7/gamma" not in bp.columns:
                continue

            bp["P7/gamma"] = pd.to_numeric(bp["P7/gamma"], errors="coerce")
            if "Timestamp" in bp.columns:
                tiempo = pd.to_numeric(bp["Timestamp"], errors="coerce")
                if tiempo.notna().any():
                    bp["t_rel"] = tiempo - tiempo.dropna().iloc[0]
                else:
                    bp["t_rel"] = range(len(bp))
            else:
                bp["t_rel"] = range(len(bp))

            DATOS[archivo.stem] = {"bp": bp}
        except (OSError, UnicodeError, pd.errors.ParserError) as error:
            print(f"No se pudo leer {archivo.name}: {error}")

    if not DATOS:
        print("No se encontraron CSV válidos con la columna P7/gamma.")
        return

    # 1.6 — Diagnóstico visual de ceros en P7/gamma
    sesiones_P7_afectadas = []
    for sid, d in DATOS.items():
        X = d['bp'][COLS_BP]
        if 'P7/gamma' in X.columns:
            n_ceros = int((X['P7/gamma'] == 0).sum())
            if n_ceros > 0:
                sesiones_P7_afectadas.append((sid, n_ceros))

    sesiones_P7_afectadas = sorted(sesiones_P7_afectadas, key=lambda x: x[1], reverse=True)

    print('SESIONES AFECTADAS EN P7/gamma')
    print('=' * 72)
    for rank, (sid, n_c) in enumerate(sesiones_P7_afectadas[:10], 1):
        print(f'   {rank}. Sesión: {sid:<30} | Cantidad de ceros: {n_c}')
    print()

    N_SESIONES_GRAF = min(3, len(sesiones_P7_afectadas))
    sids_para_graficar = [x[0] for x in sesiones_P7_afectadas[:N_SESIONES_GRAF]]

    if not sids_para_graficar:
        print('No se encontraron sesiones con ceros en la columna P7/gamma para graficar.')
    else:
        fig, axes = plt.subplots(N_SESIONES_GRAF, 1, figsize=(14, 3.0 * N_SESIONES_GRAF), squeeze=False)

        for ax, sid in zip(axes.ravel(), sids_para_graficar):
            bp = DATOS[sid]['bp']
            t = bp['t_rel']
            val = bp['P7/gamma']

            es_cero = val == 0

            ax.plot(t, val, lw=0.8, color='#1f77b4', label='Señal P7/gamma')
            ax.scatter(t[es_cero], val[es_cero], color='#d62728', s=15, zorder=3,
                       label=f'Valores cero detectados (n={es_cero.sum()})')

            ax.set_title(f'Sesión {sid} — Comportamiento temporal de P7/gamma')
            ax.set_xlabel('Tiempo transcurrido (s)')
            ax.set_ylabel('Potencia')
            ax.legend(fontsize=8, loc='upper right')

        plt.tight_layout()
        plt.show()


def mostrar_menu():
    print("\n" + "=" * 50)
    print("       MENÚ DE PROCESAMIENTO DE SEÑALES")
    print("=" * 50)
    print("1. Equipo 1 - Fast_ICA")
    print("2. Equipo 2 - Wavelet_ICA")
    print("3. Equipo 3 - Pasaband")
    print("4. Equipo 4 - InterferenciaModifier")
    print("5. Equipo 5 - ICA")
    print("6. Equipo 6 - ExtendInfomax_ICA")
    print("7. Equipo 7 - Artefacto_NoRaw")
    print("0. Salir")
    print("=" * 50)


def ejecutar_equipo(opcion):
    temas = {
        "1": ("Equipo 1", "Fast_ICA"),
        "2": ("Equipo 2", "Wavelet_ICA"),
        "3": ("Equipo 3", "Pasaband"),
        "4": ("Equipo 4", "InterferenciaModifier"),
        "5": ("Equipo 5", "ICA"),
        "6": ("Equipo 6", "ExtendInfomax_ICA"),
        "7": ("Equipo 7", "Artefacto_NoRaw")
    }

    if opcion in temas:
        equipo, tema = temas[opcion]
        print("\n" + "-" * 40)
        print(f"Seleccionaste: {equipo}")
        print(f"Tema: {tema}")
        print("-" * 40)

        if opcion == "4":
            ejecutar_equipo4()
    else:
        print("Opción no válida. Intenta nuevamente.")


def main():
    while True:
        mostrar_menu()
        opcion = input("Selecciona una opción (0-7): ").strip()

        if opcion == "0":
            print("Saliendo del programa. ¡Hasta pronto!")
            break

        ejecutar_equipo(opcion)


if __name__ == "__main__":
    main()
